In [11]:
from pymongo import MongoClient
from datetime import datetime

# Kết nối tới MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Tạo database "simplize"
db = client["simplize"]

# Danh sách các collection chính
main_collections = [
    {
        "name": "tai_chinh",
        "linh_vuc": [
            {
                "_id": "1a",
                "name": "tai_chinh_ngan_hang",
                "ma_ck": [
                    {
                        "_id": "vcb",
                        "name": "vcb",
                        "data": []  # Chúng ta sẽ thêm dữ liệu sau
                    },
                    {
                        "_id": "bid",
                        "name": "bid",
                        "data": []  # Dữ liệu sẽ được thêm sau
                    }
                ]
            },
            {"_id": "1b", "name": "chung_khoan_va_đâu_tu"},
            {"_id": "1c", "name": "bao_hiem"}
        ]
    },
    {"name": "bat_dong_san", "linh_vuc": []},
    {"name": "cong_nghiep", "linh_vuc": []},
    {"name": "hang_hoa_thiet_yeu", "linh_vuc": []},
    {"name": "hang_hoa_khong_thiet_yeu", "linh_vuc": []},
    {"name": "cong_nghe", "linh_vuc": []},
    {"name": "nguyen_vat_lieu", "linh_vuc": []},
    {"name": "nang_luong", "linh_vuc": []},
    {"name": "tien_ich", "linh_vuc": []},
    {"name": "cham_soc_suc_khoe", "linh_vuc": []},
    {"name": "dich_vu_hoc_thuat", "linh_vuc": []},
    {"name": "giao_duc", "linh_vuc": []}
]

# Tạo các collection trong database
for collection_info in main_collections:
    collection_name = collection_info["name"]
    
    # Tạo document cho mỗi collection
    document = {
        "_id": collection_name,
        "name": collection_name,
        "linh_vuc": collection_info["linh_vuc"]
    }
    
    # Chọn collection
    collection = db[collection_name]
    
    # Cập nhật hoặc chèn document
    collection.update_one(
        {"_id": collection_name},
        {"$set": document},
        upsert=True  # Nếu không tìm thấy, sẽ chèn một document mới
    )

# Dữ liệu chi tiết cho ma_ck "vcb" và "bid"
data_tai_chinh_ngan_hang = [
    {
        "Ngày": datetime(2024, 10, 1),
        "Giá mở cửa": 100,
        "Giá cao nhất": 105,
        "Giá thấp nhất": 95,
        "Giá đóng cửa": 102,
        "Thay đổi giá": 2,
        "% thay đổi": 2,
        "Khối lượng": 1000
    },
    {
        "Ngày": datetime(2024, 10, 2),
        "Giá mở cửa": 102,
        "Giá cao nhất": 107,
        "Giá thấp nhất": 99,
        "Giá đóng cửa": 104,
        "Thay đổi giá": 2,
        "% thay đổi": 2,
        "Khối lượng": 1100
    }
]
data_chung_khoan = [
    {
        "Ngày": datetime(2024, 10, 1),
        "Mã chứng khoán": "SSI",
        "Giá mở cửa": 25,
        "Giá cao nhất": 26,
        "Giá thấp nhất": 24.5,
        "Giá đóng cửa": 25.5,
        "Khối lượng": 2000
    },
    {
        "Ngày": datetime(2024, 10, 2),
        "Mã chứng khoán": "VND",
        "Giá mở cửa": 50,
        "Giá cao nhất": 52,
        "Giá thấp nhất": 49,
        "Giá đóng cửa": 51,
        "Khối lượng": 1500
    }
]
data_bao_hiem = [
    {
        "Ngày": datetime(2024, 10, 1),
        "Mã bảo hiểm": "BVH",
        "Giá mở cửa": 60,
        "Giá cao nhất": 61,
        "Giá thấp nhất": 59.5,
        "Giá đóng cửa": 60.5,
        "Khối lượng": 1000
    },
    {
        "Ngày": datetime(2024, 10, 2),
        "Mã bảo hiểm": "PTI",
        "Giá mở cửa": 45,
        "Giá cao nhất": 46,
        "Giá thấp nhất": 44.5,
        "Giá đóng cửa": 45.5,
        "Khối lượng": 1200
    }
]

# Cập nhật ma_ck "vcb" và "bid" với dữ liệu chi tiết
tai_chinh_collection = db["tai_chinh"]

# Lấy tên của linh_vuc và ma_ck
linh_vuc_name = "tai_chinh_ngan_hang"
ma_ck_names = ["vcb", "bid"]  # Cả hai ma_ck

# Lấy document của linh_vuc
tai_chinh_doc = tai_chinh_collection.find_one({"_id": "tai_chinh"})

# Kiểm tra nếu linh_vuc có tồn tại
linh_vuc = next((lv for lv in tai_chinh_doc["linh_vuc"] if lv["name"] == linh_vuc_name), None)
if linh_vuc:
    for ma_ck_name in ma_ck_names:
        # Kiểm tra nếu ma_ck có tồn tại
        ma_ck = next((ck for ck in linh_vuc["ma_ck"] if ck["name"] == ma_ck_name), None)
        if ma_ck:
            # Thực hiện cập nhật bằng cách thêm dữ liệu vào mảng "data"
            tai_chinh_collection.update_one(
                {"_id": "tai_chinh", "linh_vuc.name": linh_vuc_name, "linh_vuc.ma_ck.name": ma_ck_name},
                {
                    "$push": {
                        "linh_vuc.$[lv].ma_ck.$[ck].data": {"$each": data_tai_chinh_ngan_hang}
                    }
                },
                array_filters=[
                    {"lv.name": linh_vuc_name},
                    {"ck.name": ma_ck_name}
                ]
            )
        else:
            print(f"ma_ck '{ma_ck_name}' không tồn tại.")
else:
    print(f"linh_vuc '{linh_vuc_name}' không tồn tại.")

# Kiểm tra document đã được chèn
inserted_document = tai_chinh_collection.find_one({"_id": "tai_chinh"})

print(inserted_document)

# Đóng kết nối
client.close()


{'_id': 'tai_chinh', 'linh_vuc': [{'_id': '1a', 'name': 'tai_chinh_ngan_hang', 'ma_ck': [{'_id': 'vcb', 'name': 'vcb', 'data': [{'Ngày': datetime.datetime(2024, 10, 1, 0, 0), 'Giá mở cửa': 100, 'Giá cao nhất': 105, 'Giá thấp nhất': 95, 'Giá đóng cửa': 102, 'Thay đổi giá': 2, '% thay đổi': 2, 'Khối lượng': 1000}, {'Ngày': datetime.datetime(2024, 10, 2, 0, 0), 'Giá mở cửa': 102, 'Giá cao nhất': 107, 'Giá thấp nhất': 99, 'Giá đóng cửa': 104, 'Thay đổi giá': 2, '% thay đổi': 2, 'Khối lượng': 1100}]}, {'_id': 'bid', 'name': 'bid', 'data': [{'Ngày': datetime.datetime(2024, 10, 1, 0, 0), 'Giá mở cửa': 100, 'Giá cao nhất': 105, 'Giá thấp nhất': 95, 'Giá đóng cửa': 102, 'Thay đổi giá': 2, '% thay đổi': 2, 'Khối lượng': 1000}, {'Ngày': datetime.datetime(2024, 10, 2, 0, 0), 'Giá mở cửa': 102, 'Giá cao nhất': 107, 'Giá thấp nhất': 99, 'Giá đóng cửa': 104, 'Thay đổi giá': 2, '% thay đổi': 2, 'Khối lượng': 1100}]}]}, {'_id': '1b', 'name': 'chung_khoan_va_đâu_tu'}, {'_id': '1c', 'name': 'bao_hiem'}],